# 01 — Dataset: `Kunling/layoutlm_resume_data`
Image + Words + Bounding Boxes + NER labels (NAME, EMAIL, PHONE, ADDRESS, SECTIONHEADER).

> **Honesty note:** this notebook ships **without outputs**. Every number you see after running it is produced by the code on your machine. Nothing is pre-filled.

In [ ]:
# Run from anywhere inside the repo: move to project root so src/ imports and relative paths work
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
print("Project root:", root)

## Download once to `data/raw/layoutlm_resume_data/` (skipped if already there)

In [ ]:
!python scripts/download_dataset.py

## 1–4. Load raw dataset (from local disk), sizes, features

In [ ]:
from datasets import load_from_disk
raw = load_from_disk("data/raw/layoutlm_resume_data/arrow")
print(raw)
for split in raw:
    print(split, len(raw[split]))
print("Total:", sum(len(raw[s]) for s in raw))
print(raw["train"].features)

## Prepare the model-ready dataset (validation, box normalisation, dedup, label map)
This is the same script used from the command line.

In [ ]:
!python scripts/prepare_dataset.py

In [ ]:
from src.extraction.training import load_processed
ds, labels = load_processed()
print(ds); print("Labels:", labels)

## 5–7. Image, token and bounding-box examples

In [ ]:
ex = ds["train"][0]
print("Doc id:", ex["id"], "| words:", len(ex["words"]))
for w, b, l in list(zip(ex["words"], ex["boxes"], ex["labels"]))[:25]:
    print(f"{w:<25} {str(b):<25} {l}")
ex["image"]

## 8. Label distribution

In [ ]:
from collections import Counter
import pandas as pd
rows = []
for split in ds:
    c = Counter(l for doc in ds[split]["labels"] for l in doc)
    rows += [{"split": split, "label": k, "count": v} for k, v in c.items()]
dist = pd.DataFrame(rows).pivot(index="label", columns="split", values="count").fillna(0).astype(int)
dist

In [ ]:
ent = dist[dist.index != "O"]
ent.plot.barh(figsize=(8, 4), title="Entity token labels per split (O excluded)");

## 9. Duplicate checking (train/test leakage)

In [ ]:
import hashlib
h = {s: {hashlib.md5(" ".join(d).encode()).hexdigest() for d in ds[s]["words"]} for s in ds}
print("Unique train docs:", len(h["train"]), "/", len(ds["train"]))
print("Unique test docs:", len(h["test"]), "/", len(ds["test"]))
print("Train∩Test overlap after preparation:", len(h["train"] & h["test"]))
import json; print(json.load(open("results/dataset_stats.json")))

## 10. Visualisation: image + words + boxes + labels
Boxes are stored in the 0–1000 LayoutLMv3 space and scaled back to pixels for drawing.

In [ ]:
from PIL import ImageDraw
COLORS = {"NAME": "red", "EMAIL": "blue", "PHONE": "green", "ADDRESS": "orange", "SECTIONHEADER": "purple"}
def show(doc):
    img = doc["image"].convert("RGB"); W, H = img.size; d = ImageDraw.Draw(img)
    for w, b, l in zip(doc["words"], doc["boxes"], doc["labels"]):
        ent = l.split("-", 1)[-1] if l != "O" else None
        box = [b[0] * W / 1000, b[1] * H / 1000, b[2] * W / 1000, b[3] * H / 1000]
        d.rectangle(box, outline=COLORS.get(ent, "lightgray"), width=3 if ent else 1)
    return img
show(ds["train"][0])

In [ ]:
show(ds["test"][0])